<a href="https://colab.research.google.com/github/hrley55/BIG-DATA-NHOM-7/blob/main/Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

path_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


In [ ]:
# ====================== LOAD DATA ======================
reviews = pd.read_csv(f'{path_dir}/olist_order_reviews_dataset.csv')
orders = pd.read_csv(f'{path_dir}/olist_orders_dataset.csv')
items = pd.read_csv(f'{path_dir}/olist_order_items_dataset.csv')
payments = pd.read_csv(f'{path_dir}/olist_order_payments_dataset.csv')
customers = pd.read_csv(f'{path_dir}/olist_customers_dataset.csv')
products = pd.read_csv(f'{path_dir}/olist_products_dataset.csv')

# ====================== MERGE ======================
# 1. reviews + orders
df = reviews.merge(orders, on='order_id', how='inner')

# 2. agg items per order
items_agg = items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
).reset_index()
df = df.merge(items_agg, on='order_id', how='left')

# 3. agg payments
payments_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'mean'),
    payment_type=('payment_type', 'first')
).reset_index()
df = df.merge(payments_agg, on='order_id', how='left')

# 4. customers
df = df.merge(customers[['customer_id', 'customer_city', 'customer_state']], on='customer_id', how='left')

# 5. products (category)
products = products[['product_id', 'product_category_name']]
items = items[['order_id', 'product_id']].drop_duplicates()
items = items.merge(products, on='product_id', how='left')
category_agg = items.groupby('order_id')['product_category_name'].first().reset_index()
df = df.merge(category_agg, on='order_id', how='left')

# ====================== FEATURE ENGINEERING ======================
# Convert datetime
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

# Delivery days & is_late
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['estimated_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['is_late'] = (df['delivery_days'] > df['estimated_days']).astype(int)

# Freight ratio
df['freight_ratio'] = df['total_freight'] / (df['total_price'] + 1e-6)

# Comment length (hành vi khách hàng)
df['comment_length'] = df['review_comment_message'].str.len().fillna(0)

# Target binary
df['review_score_binary'] = np.where(df['review_score'] >= 4, 1, 0)   # 1 = Positive, 0 = Negative

# Chọn features
features = [
    'delivery_days', 'estimated_days', 'is_late', 'total_price', 'total_freight',
    'freight_ratio', 'item_count', 'payment_installments', 'payment_type',
    'customer_state', 'product_category_name', 'comment_length'
]

X = df[features].copy()
y = df['review_score_binary']

# Fill NaN
X['delivery_days'] = X['delivery_days'].fillna(X['delivery_days'].median())
X['payment_installments'] = X['payment_installments'].fillna(1)
X = X.fillna('missing')

print("Dataset shape:", X.shape)
print("Positive/Negative ratio:", y.value_counts(normalize=True))

Dataset shape: (99224, 12)
Positive/Negative ratio: review_score_binary
1    0.77068
0    0.22932
Name: proportion, dtype: float64


In [ ]:
# Preprocessing pipeline
categorical_features = ['payment_type', 'customer_state', 'product_category_name']
numeric_features = [col for col in X.columns if col not in categorical_features]

# --- START OF FIX ---
# Ensure all numeric_features are truly numeric and handle any 'missing' strings from previous fillna
for col in numeric_features:
    # Attempt to convert column to numeric, coercing any non-numeric values (like 'missing') to NaN
    X[col] = pd.to_numeric(X[col], errors='coerce')
    # If NaNs are present after coercion or from original data, fill them with the column's median
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median()) # Using median for numeric imputation
# --- END OF FIX ---

# Label encode categorical features BEFORE splitting and applying ColumnTransformer
# This is necessary because ColumnTransformer's 'passthrough' for 'cat' features expects them already processed.
for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Split 80/20 stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Now, define and apply the ColumnTransformer for scaling numeric features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', 'passthrough', categorical_features)
    ],
    remainder='passthrough' # Handle any columns not specified (shouldn't be any here)
)

# Apply the preprocessor to X_train and X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train shape (after preprocessing):", X_train_processed.shape, "Test shape (after preprocessing):", X_test_processed.shape)

# ====================== 5 MODELS ======================
models = {
    "1. LogisticRegression": LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', max_iter=1000, random_state=42),
    "2. RandomForestClassifier": RandomForestClassifier(n_estimators=200, max_depth=15, max_features='sqrt', random_state=42),
    "3. GaussianNB": GaussianNB(var_smoothing=1e-9),
    "4. SVC": SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=42),
    "5. GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.8, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"ĐANG TRAIN {name}")
    print(f"Hyperparameters: {model.get_params()}")

    # Cross-validation 5-fold using processed data
    cv_scores = cross_val_score(model, X_train_processed, y_train, cv=5, scoring='f1')
    print(f"CV F1-score (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # Train & Predict using processed data
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Test Accuracy : {acc:.4f}")
    print(f"Test F1-score : {f1:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

    results[name] = {'accuracy': acc, 'f1': f1, 'cv_f1': cv_scores.mean()}

# So sánh tổng
print("\n BẢNG SO SÁNH 5 MODELS")
print(pd.DataFrame(results).T.round(4))

Train shape (after preprocessing): (79379, 12) Test shape (after preprocessing): (19845, 12)

ĐANG TRAIN 1. LogisticRegression
Hyperparameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 1000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
CV F1-score (5-fold): 0.8907 ± 0.0009
Test Accuracy : 0.8199
Test F1-score : 0.8912

Classification Report:
               precision    recall  f1-score   support

    Negative       0.71      0.36      0.48      4551
    Positive       0.83      0.96      0.89     15294

    accuracy                           0.82     19845
   macro avg       0.77      0.66      0.68     19845
weighted avg       0.81      0.82      0.80     19845


ĐANG TRAIN 2. RandomForestClassifier
Hyperparameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'm